# Ordered Logistic Regression Results for Adoption Predictors: Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset on predictors of adoption of rangeland management interventions in Northern Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) at:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

This structured schema enables programmatic access to data records, fields, and metadata in the package.

In [ ]:
# Install mlcroissant if not already available
!pip install --quiet mlcroissant

## 1. Data Loading
In this section, we load metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata (do NOT subscript or treat as dict)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print("Dataset fields:")
for attr in ["identifier", "keywords", "dataCollectionType", "personalSensitiveInformation", "spatialCoverage", "temporalCoverage", "version", "license", "funding"]:
    val = getattr(metadata, attr, None)
    if val is not None:
        print(f"  {attr}: {val}")

## 2. Data Overview
List all available record sets and their fields using their Croissant `@id` identifiers.

**Note:** This step returns all `@id`s for record sets and their fields, so further code below references only IDs, per best practice.

In [ ]:
# Retrieve all record set IDs from the dataset metadata
record_sets = dataset.record_sets()
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}")
    # List field @ids for each record set
    field_ids = []
    if 'field' in rs:
        if isinstance(rs['field'], list):
            field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in rs['field']]
        elif isinstance(rs['field'], dict):
            field_ids = [rs['field']['@id']]
        else:
            field_ids = [rs['field']]
    print("    Fields:")
    for f_id in field_ids:
        print(f"      - {f_id}")

## 3. Data Extraction
We will load the data from each available record set, using their `@id`, into pandas DataFrames for further analysis.

Replace `<record_set_id>` with an appropriate record set `@id` from the above list.

In [ ]:
# List of available record_sets by @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Columns: {list(dataframes[record_set_id].columns)}")
        print(dataframes[record_set_id].head(2))
    else:
        print("  No records loaded.")

# Pick a record_set_id that loaded data for further EDA
nonempty_record_sets = [k for k, v in dataframes.items() if not v.empty]
if nonempty_record_sets:
    main_record_set_id = nonempty_record_sets[0]
    print(f"\nUsing main record set: {main_record_set_id}")
else:
    main_record_set_id = None
    print("No populated record sets found.")

## 4. Exploratory Data Analysis (EDA)
We now illustrate processing steps—filtering, normalization, group-by—on numeric data from the record set. All fields are referenced via their Croissant `@id`s.

Replace `<numeric_field_id>` and `<group_field_id>` below with real field IDs, as listed previously.

In [ ]:
# Select a record set to analyze
if main_record_set_id is None:
    print("No data available for EDA.")
else:
    df = dataframes[main_record_set_id]
    print(f"Working with DataFrame from record set: {main_record_set_id}")

    # Find a likely numeric field (by guessing from dtypes)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric fields found in this record set.")
        numeric_field_id = None

    # Set a threshold for filtering
    threshold = 10
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

        # Try to group by a non-numeric field
        non_numeric_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        if non_numeric_fields:
            group_field_id = non_numeric_fields[0]
            print(f"Grouping by {group_field_id}:")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(grouped_df.head())

## 5. Visualization
Let us plot the distribution of a selected numeric field from the main record set. Replace IDs with those referenced so far.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

## 6. Conclusion
In this notebook, we've shown how to:
- Access Croissant metadata and records using `mlcroissant`.
- List record sets and reference all entities by their `@id`.
- Extract data into pandas DataFrames.
- Filter and normalize numeric fields and group by categorical fields, always referencing by Croissant `@id`.
- Visualize key distributions.

This workflow enables programmatic, reproducible data exploration for FAIR and trusted research datasets.